# Aula 03 — Agente com ferramentas e ciclo de execução

**UC9 — Compreender e Aplicar Machine Learning em soluções de IA**  
**Projeto integrador:** Agentes de IA para o setor imobiliário  
**Duração:** 3 horas presenciais

## Continuação da Aula 02

Na aula anterior, construímos este fluxo:

```text
Mensagem do cliente
        ↓
Modelo aberto interpreta
        ↓
JSON
        ↓
Pydantic valida
        ↓
Python escolhe a próxima ação
```

Nesta aula, o agente deixará de apenas **escolher uma ação** e passará a **executar ferramentas**, observar os resultados e decidir o que fazer depois.

## Resultado esperado

```text
Mensagem do cliente
        ↓
Interpretação estruturada
        ↓
Agente escolhe uma ferramenta
        ↓
Python executa a ferramenta
        ↓
Resultado volta ao modelo
        ↓
Agente usa outra ferramenta ou responde
```


## Objetivos de aprendizagem

Ao final da aula, você deverá conseguir:

- explicar o que é uma ferramenta de agente;
- criar ferramentas como funções Python;
- validar os argumentos das ferramentas com Pydantic;
- disponibilizar apenas ferramentas autorizadas;
- fazer o modelo escolher uma ferramenta por meio de JSON;
- executar a ferramenta sem usar `eval`;
- devolver o resultado da ferramenta ao modelo;
- implementar um ciclo com várias etapas;
- limitar a quantidade máxima de etapas;
- registrar as decisões e resultados do agente;
- encaminhar situações sensíveis para um humano.


# 1. Preparação do ambiente

Este notebook funciona de forma independente, mas reutiliza os mesmos conceitos da Aula 02:

- modelo aberto Qwen;
- Hugging Face Transformers;
- mensagens com papéis;
- geração de JSON;
- Pydantic;
- interpretação de mensagens.

> Não utilizaremos API comercial nem chave paga.


In [ ]:
!pip install -q -U "transformers>=4.45.0" "accelerate>=0.34.0" "pydantic>=2.8.0" pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 72.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [ ]:
import json
import platform
import unicodedata
from typing import Any, Literal

import pandas as pd
import torch
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    field_validator,
    model_validator,
)
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPU disponível?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.12.13
PyTorch: 2.11.0+cpu
GPU disponível? False


# 2. Carregando o mesmo modelo da Aula 02

Usaremos:

- `Qwen/Qwen2.5-1.5B-Instruct` quando houver GPU;
- `Qwen/Qwen2.5-0.5B-Instruct` como alternativa para CPU.

O modelo de 1,5 bilhão de parâmetros tende a seguir melhor as instruções. A versão de 0,5 bilhão é mais leve, mas pode cometer mais erros ao escolher ferramentas ou produzir JSON.


In [ ]:
MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"
MODELO_CPU = "Qwen/Qwen2.5-0.5B-Instruct"

MODELO_ID = MODELO_GPU if torch.cuda.is_available() else MODELO_CPU

print("Modelo selecionado:", MODELO_ID)


Modelo selecionado: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype="auto",
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()

dispositivo_modelo = next(model.parameters()).device

print("Modelo carregado.")
print("Dispositivo principal:", dispositivo_modelo)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado.
Dispositivo principal: cpu


## Função de geração

A função utiliza o template de conversa do próprio modelo.

Ela recebe mensagens nos papéis `system`, `user` e `assistant` e devolve apenas os novos tokens produzidos pelo modelo.


In [ ]:
def gerar_resposta(
    mensagens: list[dict[str, str]],
    max_novos_tokens: int = 350,
) -> str:
    """Gera uma resposta usando o modelo aberto carregado no notebook."""

    entradas = tokenizer.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    entradas = {
        nome: tensor.to(dispositivo_modelo)
        for nome, tensor in entradas.items()
    }

    with torch.inference_mode():
        saidas = model.generate(
            **entradas,
            max_new_tokens=max_novos_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    quantidade_tokens_entrada = entradas["input_ids"].shape[1]
    novos_tokens = saidas[0][quantidade_tokens_entrada:]

    return tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True,
    ).strip()


# 3. Reaproveitando a interpretação da Aula 02

Primeiro, o modelo transforma a mensagem do cliente em uma estrutura de qualificação.

Essa etapa não executa ferramentas. Ela apenas interpreta a linguagem natural.


In [ ]:
class QualificacaoCliente(BaseModel):
    model_config = ConfigDict(extra="forbid")

    finalidade: Literal["compra", "locacao", "nao_informado"]
    tipo_imovel: str | None = None
    regiao_desejada: str | None = None
    orcamento_maximo: float | None = None
    quantidade_quartos: int | None = None
    precisa_financiamento: bool | None = None
    assunto_sensivel: bool = False
    resumo: str = Field(min_length=1, max_length=250)

    @field_validator("orcamento_maximo")
    @classmethod
    def validar_orcamento(cls, valor: float | None) -> float | None:
        if valor is not None and valor <= 0:
            raise ValueError("O orçamento deve ser maior que zero.")
        return valor

    @field_validator("quantidade_quartos")
    @classmethod
    def validar_quartos(cls, valor: int | None) -> int | None:
        if valor is not None and valor <= 0:
            raise ValueError("A quantidade de quartos deve ser maior que zero.")
        return valor


In [ ]:
INSTRUCAO_EXTRACAO = '''
Você é um componente de extração de informações de uma imobiliária.

Analise a mensagem do cliente e responda SOMENTE com um objeto JSON válido.
Não use Markdown, não use ``` e não escreva explicações fora do JSON.

Use exatamente estas chaves:
{
  "finalidade": "compra | locacao | nao_informado",
  "tipo_imovel": "texto ou null",
  "regiao_desejada": "texto ou null",
  "orcamento_maximo": "número ou null",
  "quantidade_quartos": "inteiro ou null",
  "precisa_financiamento": "true, false ou null",
  "assunto_sensivel": "true ou false",
  "resumo": "resumo curto em português"
}

Regras:
1. Não invente informações.
2. Use null quando a informação não estiver presente.
3. Converta expressões como "450 mil" para 450000.
4. Use "locacao", sem cedilha ou acento, no campo finalidade.
5. Marque assunto_sensivel como true quando o cliente pedir orientação
   jurídica, contratual ou análise financeira detalhada.
'''


In [ ]:
def extrair_json(texto: str) -> dict:
    """Extrai o primeiro objeto JSON válido encontrado em um texto."""

    texto_limpo = (
        texto.strip()
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .strip()
    )

    inicio = texto_limpo.find("{")
    if inicio == -1:
        raise ValueError("Nenhum objeto JSON foi encontrado na resposta.")

    decodificador = json.JSONDecoder()
    objeto, _ = decodificador.raw_decode(texto_limpo[inicio:])

    if not isinstance(objeto, dict):
        raise ValueError("A resposta encontrada não é um objeto JSON.")

    return objeto


def interpretar_mensagem_cliente(
    mensagem: str,
) -> tuple[QualificacaoCliente | None, str]:
    """Interpreta a mensagem do cliente e valida a resposta."""

    mensagens = [
        {"role": "system", "content": INSTRUCAO_EXTRACAO},
        {"role": "user", "content": mensagem},
    ]

    resposta = gerar_resposta(
        mensagens,
        max_novos_tokens=250,
    )

    try:
        objeto_json = extrair_json(resposta)
        qualificacao = QualificacaoCliente.model_validate(objeto_json)
        return qualificacao, resposta

    except (ValueError, json.JSONDecodeError, ValidationError) as erro:
        print("Não foi possível validar a interpretação.")
        print("Tipo do erro:", type(erro).__name__)
        print("Detalhes:", erro)
        return None, resposta


# 4. Criando uma base fictícia de imóveis

Uma ferramenta precisa acessar algum recurso real da aplicação.

Nesta aula, utilizaremos uma base fictícia em memória. Em um projeto maior, essa mesma função poderia consultar:

- arquivo CSV;
- banco SQLite;
- API de imóveis;
- sistema interno da imobiliária.

Os dados abaixo são apenas didáticos.


In [ ]:
dados_imoveis = [
    {
        "id": 1,
        "finalidade": "compra",
        "tipo_imovel": "apartamento",
        "regiao": "Águas Claras",
        "quartos": 2,
        "vagas": 1,
        "area_m2": 58,
        "preco": 430000,
    },
    {
        "id": 2,
        "finalidade": "compra",
        "tipo_imovel": "apartamento",
        "regiao": "Águas Claras",
        "quartos": 2,
        "vagas": 1,
        "area_m2": 64,
        "preco": 470000,
    },
    {
        "id": 3,
        "finalidade": "compra",
        "tipo_imovel": "apartamento",
        "regiao": "Águas Claras",
        "quartos": 3,
        "vagas": 2,
        "area_m2": 85,
        "preco": 520000,
    },
    {
        "id": 4,
        "finalidade": "compra",
        "tipo_imovel": "casa",
        "regiao": "Taguatinga",
        "quartos": 3,
        "vagas": 2,
        "area_m2": 120,
        "preco": 510000,
    },
    {
        "id": 5,
        "finalidade": "compra",
        "tipo_imovel": "apartamento",
        "regiao": "Asa Norte",
        "quartos": 2,
        "vagas": 1,
        "area_m2": 70,
        "preco": 680000,
    },
    {
        "id": 6,
        "finalidade": "compra",
        "tipo_imovel": "kitnet",
        "regiao": "Asa Norte",
        "quartos": 1,
        "vagas": 0,
        "area_m2": 30,
        "preco": 290000,
    },
    {
        "id": 7,
        "finalidade": "compra",
        "tipo_imovel": "casa",
        "regiao": "Ceilândia",
        "quartos": 3,
        "vagas": 1,
        "area_m2": 100,
        "preco": 390000,
    },
    {
        "id": 8,
        "finalidade": "locacao",
        "tipo_imovel": "apartamento",
        "regiao": "Águas Claras",
        "quartos": 2,
        "vagas": 1,
        "area_m2": 62,
        "preco": 2800,
    },
    {
        "id": 9,
        "finalidade": "locacao",
        "tipo_imovel": "apartamento",
        "regiao": "Águas Claras",
        "quartos": 3,
        "vagas": 2,
        "area_m2": 84,
        "preco": 3600,
    },
    {
        "id": 10,
        "finalidade": "locacao",
        "tipo_imovel": "kitnet",
        "regiao": "Asa Norte",
        "quartos": 1,
        "vagas": 0,
        "area_m2": 28,
        "preco": 1900,
    },
    {
        "id": 11,
        "finalidade": "locacao",
        "tipo_imovel": "casa",
        "regiao": "Taguatinga",
        "quartos": 3,
        "vagas": 2,
        "area_m2": 110,
        "preco": 3200,
    },
    {
        "id": 12,
        "finalidade": "locacao",
        "tipo_imovel": "apartamento",
        "regiao": "Samambaia",
        "quartos": 2,
        "vagas": 1,
        "area_m2": 55,
        "preco": 2100,
    },
]

df_imoveis = pd.DataFrame(dados_imoveis)

# O CSV poderá ser utilizado por outras aulas.
df_imoveis.to_csv("imoveis_ficticios.csv", index=False)

df_imoveis


,id,finalidade,tipo_imovel,regiao,quartos,vagas,area_m2,preco
0,1,compra,apartamento,Águas Claras,2,1,58,430000
1,2,compra,apartamento,Águas Claras,2,1,64,470000
2,3,compra,apartamento,Águas Claras,3,2,85,520000
3,4,compra,casa,Taguatinga,3,2,120,510000
4,5,compra,apartamento,Asa Norte,2,1,70,680000
5,6,compra,kitnet,Asa Norte,1,0,30,290000
6,7,compra,casa,Ceilândia,3,1,100,390000
7,8,locacao,apartamento,Águas Claras,2,1,62,2800
8,9,locacao,apartamento,Águas Claras,3,2,84,3600
9,10,locacao,kitnet,Asa Norte,1,0,28,1900


# 5. O que é uma ferramenta?

Uma ferramenta é uma função controlada que o agente pode solicitar.

Exemplos nesta aula:

1. `consultar_imoveis`: pesquisa a base fictícia.
2. `calcular_entrada`: calcula uma estimativa simples de entrada.

O modelo **não executa Python diretamente**. Ele produz uma solicitação estruturada. O nosso programa valida a solicitação e decide se executará a função.


## Modelos de argumentos

Cada ferramenta terá um modelo Pydantic próprio.

Isso impede chamadas como:

```json
{
  "percentual_entrada": -200,
  "valor_imovel": "qualquer coisa"
}
```


In [ ]:
class ConsultaImoveisArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")

    finalidade: Literal["compra", "locacao"]
    tipo_imovel: str
    regiao_desejada: str
    orcamento_maximo: float = Field(gt=0)
    quantidade_quartos: int | None = Field(default=None, gt=0)
    limite: int = Field(default=3, ge=1, le=5)


class CalculoEntradaArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")

    valor_imovel: float = Field(gt=0)
    percentual_entrada: float = Field(gt=0, le=100)


## Ferramenta 1 — Consultar imóveis

A função:

- normaliza textos;
- filtra finalidade;
- filtra tipo;
- filtra região;
- respeita o orçamento máximo;
- considera a quantidade mínima de quartos;
- devolve os imóveis mais baratos primeiro.


In [ ]:
def normalizar_texto(valor: str) -> str:
    """Remove acentos e converte o texto para minúsculas."""

    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(
        caractere
        for caractere in valor
        if not unicodedata.combining(caractere)
    )
    return valor.strip().lower()


def consultar_imoveis(
    finalidade: str,
    tipo_imovel: str,
    regiao_desejada: str,
    orcamento_maximo: float,
    quantidade_quartos: int | None = None,
    limite: int = 3,
) -> dict:
    """Consulta a base fictícia conforme os filtros informados."""

    resultado = df_imoveis.copy()

    resultado = resultado[
        resultado["finalidade"].map(normalizar_texto)
        == normalizar_texto(finalidade)
    ]

    resultado = resultado[
        resultado["tipo_imovel"].map(normalizar_texto)
        == normalizar_texto(tipo_imovel)
    ]

    resultado = resultado[
        resultado["regiao"].map(normalizar_texto)
        == normalizar_texto(regiao_desejada)
    ]

    resultado = resultado[
        resultado["preco"] <= orcamento_maximo
    ]

    if quantidade_quartos is not None:
        resultado = resultado[
            resultado["quartos"] >= quantidade_quartos
        ]

    resultado = (
        resultado
        .sort_values("preco")
        .head(limite)
    )

    # Converte o DataFrame em tipos simples compatíveis com JSON.
    imoveis = json.loads(
        resultado.to_json(
            orient="records",
            force_ascii=False,
        )
    )

    return {
        "quantidade_encontrada": len(imoveis),
        "imoveis": imoveis,
    }


In [ ]:
# Teste direto da ferramenta, sem utilizar o modelo.

teste_consulta = consultar_imoveis(
    finalidade="compra",
    tipo_imovel="apartamento",
    regiao_desejada="Águas Claras",
    orcamento_maximo=500000,
    quantidade_quartos=2,
)

print(json.dumps(teste_consulta, ensure_ascii=False, indent=2))


{
  "quantidade_encontrada": 2,
  "imoveis": [
    {
      "id": 1,
      "finalidade": "compra",
      "tipo_imovel": "apartamento",
      "regiao": "Águas Claras",
      "quartos": 2,
      "vagas": 1,
      "area_m2": 58,
      "preco": 430000
    },
    {
      "id": 2,
      "finalidade": "compra",
      "tipo_imovel": "apartamento",
      "regiao": "Águas Claras",
      "quartos": 2,
      "vagas": 1,
      "area_m2": 64,
      "preco": 470000
    }
  ]
}


## Ferramenta 2 — Calcular entrada

Esta ferramenta faz apenas uma estimativa matemática didática.

Ela não calcula juros, financiamento, taxas bancárias ou condições contratuais.


In [ ]:
def calcular_entrada(
    valor_imovel: float,
    percentual_entrada: float,
) -> dict:
    """Calcula uma estimativa simples de entrada e saldo restante."""

    valor_entrada = valor_imovel * percentual_entrada / 100
    saldo_restante = valor_imovel - valor_entrada

    return {
        "valor_imovel": round(valor_imovel, 2),
        "percentual_entrada": round(percentual_entrada, 2),
        "valor_entrada": round(valor_entrada, 2),
        "saldo_restante": round(saldo_restante, 2),
        "observacao": (
            "Estimativa matemática didática. "
            "Não representa proposta de financiamento."
        ),
    }


In [ ]:
teste_calculo = calcular_entrada(
    valor_imovel=430000,
    percentual_entrada=20,
)

print(json.dumps(teste_calculo, ensure_ascii=False, indent=2))


{
  "valor_imovel": 430000,
  "percentual_entrada": 20,
  "valor_entrada": 86000.0,
  "saldo_restante": 344000.0,
  "observacao": "Estimativa matemática didática. Não representa proposta de financiamento."
}


# 6. Registro autorizado de ferramentas

Não permitiremos que o modelo escolha qualquer função do Python.

Usaremos uma **lista autorizada** contendo:

- nome público;
- função Python;
- modelo de validação;
- descrição.

> Nunca use `eval()` para executar uma ação criada pelo modelo.


In [ ]:
FERRAMENTAS = {
    "consultar_imoveis": {
        "funcao": consultar_imoveis,
        "modelo_argumentos": ConsultaImoveisArgs,
        "descricao": (
            "Pesquisa imóveis na base fictícia. "
            "Use quando os dados mínimos do cliente estiverem disponíveis."
        ),
    },
    "calcular_entrada": {
        "funcao": calcular_entrada,
        "modelo_argumentos": CalculoEntradaArgs,
        "descricao": (
            "Calcula uma estimativa simples de entrada. "
            "Use somente quando houver valor do imóvel e percentual."
        ),
    },
}

list(FERRAMENTAS.keys())


['consultar_imoveis', 'calcular_entrada']

In [ ]:
def executar_ferramenta(
    nome: str,
    argumentos: dict[str, Any],
) -> dict:
    """Valida e executa somente ferramentas registradas."""

    configuracao = FERRAMENTAS.get(nome)

    if configuracao is None:
        return {
            "ok": False,
            "erro": f"Ferramenta não autorizada: {nome}",
        }

    modelo_argumentos = configuracao["modelo_argumentos"]
    funcao = configuracao["funcao"]

    try:
        argumentos_validados = modelo_argumentos.model_validate(argumentos)

        resultado = funcao(
            **argumentos_validados.model_dump()
        )

        return {
            "ok": True,
            "ferramenta": nome,
            "resultado": resultado,
        }

    except ValidationError as erro:
        return {
            "ok": False,
            "ferramenta": nome,
            "erro": "Argumentos inválidos.",
            "detalhes": erro.errors(),
        }

    except Exception as erro:
        return {
            "ok": False,
            "ferramenta": nome,
            "erro": "Falha inesperada ao executar a ferramenta.",
            "detalhes": str(erro),
        }


In [ ]:
# Testando o executor autorizado.

execucao_valida = executar_ferramenta(
    "calcular_entrada",
    {
        "valor_imovel": 430000,
        "percentual_entrada": 20,
    },
)

print(json.dumps(execucao_valida, ensure_ascii=False, indent=2))


{
  "ok": true,
  "ferramenta": "calcular_entrada",
  "resultado": {
    "valor_imovel": 430000.0,
    "percentual_entrada": 20.0,
    "valor_entrada": 86000.0,
    "saldo_restante": 344000.0,
    "observacao": "Estimativa matemática didática. Não representa proposta de financiamento."
  }
}


In [ ]:
# O Pydantic rejeita argumentos perigosos ou incoerentes.

execucao_invalida = executar_ferramenta(
    "calcular_entrada",
    {
        "valor_imovel": 430000,
        "percentual_entrada": -50,
        "comando_extra": "apagar banco",
    },
)

print(json.dumps(execucao_invalida, ensure_ascii=False, indent=2))


{
  "ok": false,
  "ferramenta": "calcular_entrada",
  "erro": "Argumentos inválidos.",
  "detalhes": [
    {
      "type": "greater_than",
      "loc": [
        "percentual_entrada"
      ],
      "msg": "Input should be greater than 0",
      "input": -50,
      "ctx": {
        "gt": 0.0
      },
      "url": "https://errors.pydantic.dev/2.13/v/greater_than"
    },
    {
      "type": "extra_forbidden",
      "loc": [
        "comando_extra"
      ],
      "msg": "Extra inputs are not permitted",
      "input": "apagar banco",
      "url": "https://errors.pydantic.dev/2.13/v/extra_forbidden"
    }
  ]
}


# 7. Como o modelo solicitará uma ação

O modelo deverá devolver um JSON em um destes formatos.

### Usar ferramenta

```json
{
  "tipo": "usar_ferramenta",
  "ferramenta": "consultar_imoveis",
  "argumentos": {
    "finalidade": "compra",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": 500000,
    "quantidade_quartos": 2,
    "limite": 3
  },
  "resposta": null
}
```

### Responder ao cliente

```json
{
  "tipo": "responder",
  "ferramenta": null,
  "argumentos": {},
  "resposta": "Encontrei duas opções compatíveis."
}
```

### Encaminhar para humano

```json
{
  "tipo": "encaminhar_humano",
  "ferramenta": null,
  "argumentos": {},
  "resposta": "Essa solicitação precisa de análise profissional."
}
```


In [ ]:
class AcaoAgente(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tipo: Literal[
        "usar_ferramenta",
        "responder",
        "encaminhar_humano",
    ]
    ferramenta: Literal[
        "consultar_imoveis",
        "calcular_entrada",
    ] | None = None
    argumentos: dict[str, Any] = Field(default_factory=dict)
    resposta: str | None = None

    @model_validator(mode="after")
    def validar_coerencia(self):
        if self.tipo == "usar_ferramenta":
            if self.ferramenta is None:
                raise ValueError(
                    "Uma ação de ferramenta precisa informar a ferramenta."
                )

        if self.tipo in {"responder", "encaminhar_humano"}:
            if not self.resposta:
                raise ValueError(
                    "A resposta é obrigatória ao finalizar ou encaminhar."
                )

        return self


# 8. Instrução do agente

O agente receberá:

- mensagem original;
- qualificação estruturada;
- ferramentas disponíveis;
- observações das ferramentas já executadas.

Ele deverá escolher **somente uma ação por etapa**.


In [ ]:
INSTRUCAO_AGENTE = '''
Você é um agente imobiliário educacional.

Você recebe:
1. a mensagem original do cliente;
2. uma qualificação estruturada;
3. resultados de ferramentas já executadas.

Sua tarefa é escolher exatamente UMA próxima ação.

Você pode:
- usar a ferramenta consultar_imoveis;
- usar a ferramenta calcular_entrada;
- responder ao cliente;
- encaminhar para humano.

Responda SOMENTE com um objeto JSON válido, sem Markdown e sem explicações.

Formato obrigatório:
{
  "tipo": "usar_ferramenta | responder | encaminhar_humano",
  "ferramenta": "consultar_imoveis | calcular_entrada | null",
  "argumentos": {},
  "resposta": "texto ou null"
}

REGRAS:
1. Nunca invente imóveis ou valores.
2. Use somente os resultados recebidos das ferramentas.
3. Use consultar_imoveis quando houver finalidade, tipo, região e orçamento.
4. Se faltarem dados mínimos, responda com uma pergunta curta ao cliente.
5. Use calcular_entrada somente quando o cliente pedir uma estimativa
   e já existir um valor de imóvel.
6. Quando houver vários imóveis, o primeiro resultado é o mais barato.
7. Se a solicitação for jurídica, contratual ou financeira detalhada,
   encaminhe para humano.
8. Escolha uma única ação em cada etapa.
9. Não repita uma ferramenta se o resultado dela já estiver disponível.
10. Depois de obter os resultados necessários, responda ao cliente.
'''


In [ ]:
DESCRICAO_FERRAMENTAS = {
    nome: {
        "descricao": dados["descricao"],
        "schema_argumentos": dados["modelo_argumentos"].model_json_schema(),
    }
    for nome, dados in FERRAMENTAS.items()
}

print(
    json.dumps(
        DESCRICAO_FERRAMENTAS,
        ensure_ascii=False,
        indent=2,
    )
)


{
  "consultar_imoveis": {
    "descricao": "Pesquisa imóveis na base fictícia. Use quando os dados mínimos do cliente estiverem disponíveis.",
    "schema_argumentos": {
      "additionalProperties": false,
      "properties": {
        "finalidade": {
          "enum": [
            "compra",
            "locacao"
          ],
          "title": "Finalidade",
          "type": "string"
        },
        "tipo_imovel": {
          "title": "Tipo Imovel",
          "type": "string"
        },
        "regiao_desejada": {
          "title": "Regiao Desejada",
          "type": "string"
        },
        "orcamento_maximo": {
          "exclusiveMinimum": 0,
          "title": "Orcamento Maximo",
          "type": "number"
        },
        "quantidade_quartos": {
          "anyOf": [
            {
              "exclusiveMinimum": 0,
              "type": "integer"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
        

# 9. Fazendo o modelo escolher uma ação

Primeiro, vamos executar apenas uma etapa, sem ciclo.


In [ ]:
def escolher_acao(
    mensagem_cliente: str,
    qualificacao: QualificacaoCliente,
    historico_execucao: list[dict[str, Any]],
) -> tuple[AcaoAgente | None, str]:
    """Solicita ao modelo uma única próxima ação."""

    contexto = {
        "mensagem_cliente": mensagem_cliente,
        "qualificacao": qualificacao.model_dump(),
        "ferramentas_disponiveis": DESCRICAO_FERRAMENTAS,
        "historico_execucao": historico_execucao,
    }

    mensagens = [
        {
            "role": "system",
            "content": INSTRUCAO_AGENTE,
        },
        {
            "role": "user",
            "content": json.dumps(
                contexto,
                ensure_ascii=False,
                indent=2,
            ),
        },
    ]

    resposta = gerar_resposta(
        mensagens,
        max_novos_tokens=350,
    )

    try:
        objeto = extrair_json(resposta)
        acao = AcaoAgente.model_validate(objeto)
        return acao, resposta

    except (ValueError, json.JSONDecodeError, ValidationError) as erro:
        print("Não foi possível validar a ação do agente.")
        print("Tipo do erro:", type(erro).__name__)
        print("Detalhes:", erro)
        return None, resposta


In [ ]:
mensagem_exemplo = (
    "Quero comprar um apartamento de dois quartos em Águas Claras "
    "até R$ 500 mil."
)

qualificacao_exemplo, resposta_extracao = interpretar_mensagem_cliente(
    mensagem_exemplo
)

if qualificacao_exemplo:
    acao_exemplo, resposta_acao = escolher_acao(
        mensagem_cliente=mensagem_exemplo,
        qualificacao=qualificacao_exemplo,
        historico_execucao=[],
    )

    print("Qualificação:")
    print(qualificacao_exemplo.model_dump_json(indent=2))

    print("\nAção escolhida:")
    print(acao_exemplo)

    print("\nResposta bruta do modelo:")
    print(resposta_acao)


Não foi possível validar a ação do agente.
Tipo do erro: ValidationError
Detalhes: 1 validation error for AcaoAgente
tipo
  Input should be 'usar_ferramenta', 'responder' or 'encaminhar_humano' [type=literal_error, input_value='calcular_entrada', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Qualificação:
{
  "finalidade": "compra",
  "tipo_imovel": "apartamento",
  "regiao_desejada": "Águas Claras",
  "orcamento_maximo": 500000.0,
  "quantidade_quartos": 2,
  "precisa_financiamento": true,
  "assunto_sensivel": true,
  "resumo": "O apartamento está disponível em Águas Claras, com 2 quartos, e tem um orçamento máximo de R$ 500 mil."
}

Ação escolhida:
None

Resposta bruta do modelo:
{
  "tipo": "calcular_entrada",
  "ferramenta": "calcular_entrada",
  "argumentos": {
    "valor_imovel": 500000.0,
    "percentual_entrada": 50
  },
  "resposta": "Agora que você informou os valores do imóvel e o percentual de entrada, podemos calcular u

# 10. Executando uma etapa completa

Uma etapa do agente contém:

```text
Modelo escolhe ação
        ↓
Pydantic valida
        ↓
Python executa ferramenta
        ↓
Resultado é registrado como observação
```


In [ ]:
if (
    qualificacao_exemplo
    and acao_exemplo
    and acao_exemplo.tipo == "usar_ferramenta"
):
    observacao_exemplo = executar_ferramenta(
        acao_exemplo.ferramenta,
        acao_exemplo.argumentos,
    )

    print(
        json.dumps(
            observacao_exemplo,
            ensure_ascii=False,
            indent=2,
        )
    )


# 11. Criando o ciclo completo do agente

Agora faremos o agente repetir o processo.

A cada etapa:

1. o modelo escolhe uma ação;
2. a ação é validada;
3. a ferramenta é executada;
4. o resultado volta para o histórico;
5. o modelo decide novamente.

O ciclo termina quando:

- o modelo responde;
- o modelo encaminha para humano;
- ocorre um erro;
- o limite máximo de etapas é alcançado.


In [ ]:
def executar_agente(
    mensagem_cliente: str,
    max_etapas: int = 4,
) -> dict:
    """Executa o agente com ferramentas e limite de etapas."""

    qualificacao, resposta_extracao = interpretar_mensagem_cliente(
        mensagem_cliente
    )

    if qualificacao is None:
        return {
            "status": "erro_interpretacao",
            "resposta": (
                "Não consegui interpretar a mensagem com segurança. "
                "Solicite que o cliente reformule."
            ),
            "resposta_bruta": resposta_extracao,
            "etapas": [],
        }

    if qualificacao.assunto_sensivel:
        return {
            "status": "encaminhar_humano",
            "resposta": (
                "Essa solicitação precisa ser analisada "
                "por um profissional responsável."
            ),
            "qualificacao": qualificacao.model_dump(),
            "etapas": [],
        }

    historico_execucao = []

    for numero_etapa in range(1, max_etapas + 1):
        acao, resposta_bruta = escolher_acao(
            mensagem_cliente=mensagem_cliente,
            qualificacao=qualificacao,
            historico_execucao=historico_execucao,
        )

        registro = {
            "etapa": numero_etapa,
            "resposta_bruta_modelo": resposta_bruta,
        }

        if acao is None:
            registro["erro"] = "Ação inválida."
            historico_execucao.append(registro)

            return {
                "status": "erro_acao",
                "resposta": (
                    "O agente produziu uma ação inválida "
                    "e interrompeu a execução."
                ),
                "qualificacao": qualificacao.model_dump(),
                "etapas": historico_execucao,
            }

        registro["acao"] = acao.model_dump()

        if acao.tipo == "responder":
            historico_execucao.append(registro)

            return {
                "status": "concluido",
                "resposta": acao.resposta,
                "qualificacao": qualificacao.model_dump(),
                "etapas": historico_execucao,
            }

        if acao.tipo == "encaminhar_humano":
            historico_execucao.append(registro)

            return {
                "status": "encaminhar_humano",
                "resposta": acao.resposta,
                "qualificacao": qualificacao.model_dump(),
                "etapas": historico_execucao,
            }

        observacao = executar_ferramenta(
            nome=acao.ferramenta,
            argumentos=acao.argumentos,
        )

        registro["observacao_ferramenta"] = observacao
        historico_execucao.append(registro)

        if not observacao["ok"]:
            return {
                "status": "erro_ferramenta",
                "resposta": (
                    "A ferramenta não pôde ser executada "
                    "com os argumentos fornecidos."
                ),
                "qualificacao": qualificacao.model_dump(),
                "etapas": historico_execucao,
            }

    return {
        "status": "limite_etapas",
        "resposta": (
            "O agente atingiu o limite máximo de etapas "
            "e interrompeu a execução por segurança."
        ),
        "qualificacao": qualificacao.model_dump(),
        "etapas": historico_execucao,
    }


## Primeiro teste completo

O agente deverá:

1. interpretar a mensagem;
2. consultar a base;
3. receber os imóveis;
4. responder usando somente os resultados.


In [ ]:
resultado_agente = executar_agente(
    "Quero comprar um apartamento de dois quartos em Águas Claras "
    "até R$ 500 mil.",
    max_etapas=4,
)

print(
    json.dumps(
        resultado_agente,
        ensure_ascii=False,
        indent=2,
    )
)


{
  "status": "encaminhar_humano",
  "resposta": "Essa solicitação precisa ser analisada por um profissional responsável.",
  "qualificacao": {
    "finalidade": "compra",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": 500000.0,
    "quantidade_quartos": 2,
    "precisa_financiamento": true,
    "assunto_sensivel": true,
    "resumo": "O apartamento está disponível em Águas Claras, com 2 quartos, e tem um orçamento máximo de R$ 500 mil."
  },
  "etapas": []
}


## Teste com duas ferramentas

Neste cenário, o cliente também pede o cálculo de 20% de entrada para o imóvel mais barato.

Um fluxo possível é:

```text
Etapa 1: consultar imóveis
Etapa 2: calcular entrada do primeiro imóvel
Etapa 3: responder ao cliente
```


In [ ]:
resultado_duas_ferramentas = executar_agente(
    (
        "Quero comprar um apartamento de dois quartos em Águas Claras "
        "até R$ 500 mil. Para a opção mais barata, calcule uma entrada de 20%."
    ),
    max_etapas=4,
)

print(
    json.dumps(
        resultado_duas_ferramentas,
        ensure_ascii=False,
        indent=2,
    )
)


Não foi possível validar a interpretação.
Tipo do erro: ValidationError
Detalhes: 1 validation error for QualificacaoCliente
orcamento_maximo
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='20%', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing
{
  "status": "erro_interpretacao",
  "resposta": "Não consegui interpretar a mensagem com segurança. Solicite que o cliente reformule.",
  "resposta_bruta": "{\n  \"finalidade\": \"compra\",\n  \"tipo_imovel\": \"apartamento\",\n  \"regiao_desejada\": \"Águas Claras\",\n  \"orcamento_maximo\": \"20%\",\n  \"quantidade_quartos\": 2,\n  \"precisa_financiamento\": true,\n  \"assunto_sensivel\": \"false\",\n  \"resumo\": \"Aqui está a sua opção favorita: um apartamento de dois quartos em Águas Claras, com um orçamento de 20%.\"\n}",
  "etapas": []
}


# 12. Visualizando o registro das etapas

O histórico ajuda a responder perguntas como:

- qual ferramenta foi escolhida;
- quais argumentos foram enviados;
- qual resultado a ferramenta devolveu;
- quantas etapas foram utilizadas;
- em que ponto ocorreu um erro.


In [ ]:
def criar_tabela_etapas(resultado: dict) -> pd.DataFrame:
    linhas = []

    for etapa in resultado.get("etapas", []):
        acao = etapa.get("acao", {})
        observacao = etapa.get("observacao_ferramenta", {})

        linhas.append(
            {
                "etapa": etapa.get("etapa"),
                "tipo_acao": acao.get("tipo"),
                "ferramenta": acao.get("ferramenta"),
                "argumentos": json.dumps(
                    acao.get("argumentos", {}),
                    ensure_ascii=False,
                ),
                "execucao_ok": observacao.get("ok"),
                "erro": etapa.get("erro")
                or observacao.get("erro"),
            }
        )

    return pd.DataFrame(linhas)


tabela_etapas = criar_tabela_etapas(
    resultado_duas_ferramentas
)

tabela_etapas


""


# 13. Cenários de teste

Teste situações diferentes:

1. dados completos para consulta;
2. informação obrigatória ausente;
3. nenhum imóvel encontrado;
4. consulta de locação;
5. pergunta jurídica;
6. solicitação de cálculo;
7. mensagem ambígua.

Modelos pequenos podem falhar. O objetivo é identificar onde precisamos de melhores instruções, validações ou regras.


In [ ]:
cenarios = [
    "Quero alugar um apartamento de 2 quartos em Águas Claras até R$ 3 mil.",
    "Quero comprar um apartamento de 2 quartos.",
    "Procuro uma casa de 5 quartos na Asa Norte até R$ 400 mil.",
    "Quero comprar uma casa de 3 quartos em Taguatinga até R$ 550 mil.",
    "Analise o contrato deste imóvel e diga se posso processar a imobiliária.",
]

resultados_testes = []

for numero, mensagem in enumerate(cenarios, start=1):
    print("\n" + "=" * 90)
    print(f"CENÁRIO {numero}: {mensagem}")

    resultado = executar_agente(
        mensagem,
        max_etapas=4,
    )

    print("Status:", resultado["status"])
    print("Resposta:", resultado["resposta"])

    resultados_testes.append(
        {
            "cenario": numero,
            "mensagem": mensagem,
            "status": resultado["status"],
            "resposta": resultado["resposta"],
            "quantidade_etapas": len(resultado.get("etapas", [])),
        }
    )

pd.DataFrame(resultados_testes)



CENÁRIO 1: Quero alugar um apartamento de 2 quartos em Águas Claras até R$ 3 mil.
Não foi possível validar a ação do agente.
Tipo do erro: ValidationError
Detalhes: 1 validation error for AcaoAgente
ferramenta
  Input should be 'consultar_imoveis' or 'calcular_entrada' [type=literal_error, input_value='encaminhar_humano', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Status: erro_acao
Resposta: O agente produziu uma ação inválida e interrompeu a execução.

CENÁRIO 2: Quero comprar um apartamento de 2 quartos.
Não foi possível validar a ação do agente.
Tipo do erro: ValidationError
Detalhes: 1 validation error for AcaoAgente
tipo
  Input should be 'usar_ferramenta', 'responder' or 'encaminhar_humano' [type=literal_error, input_value='calcular_entrada', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Status: erro_acao
Resposta: O agente produziu uma ação inválida e interrompeu a execu

,cenario,mensagem,status,resposta,quantidade_etapas
0,1,Quero alugar um apartamento de 2 quartos em Ág...,erro_acao,O agente produziu uma ação inválida e interrom...,1
1,2,Quero comprar um apartamento de 2 quartos.,erro_acao,O agente produziu uma ação inválida e interrom...,1
2,3,Procuro uma casa de 5 quartos na Asa Norte até...,encaminhar_humano,Essa solicitação precisa ser analisada por um ...,0
3,4,Quero comprar uma casa de 3 quartos em Taguati...,encaminhar_humano,Essa solicitação precisa ser analisada por um ...,0
4,5,Analise o contrato deste imóvel e diga se poss...,erro_interpretacao,Não consegui interpretar a mensagem com segura...,0


# 14. Proteções implementadas

Nosso protótipo já possui algumas proteções:

- ferramentas em lista autorizada;
- argumentos validados com Pydantic;
- nenhum uso de `eval`;
- nenhuma execução de código produzido pelo modelo;
- limite máximo de etapas;
- interrupção em caso de erro;
- encaminhamento de assuntos sensíveis;
- registro das decisões;
- base fictícia;
- cálculo identificado como estimativa didática.

Essas proteções não tornam o protótipo pronto para produção, mas ajudam a demonstrar uma arquitetura mais segura.


# Atividade prática — Parte 1

Em dupla ou trio:

1. Execute os dois cenários principais.
2. Verifique se o agente:
   - consulta a base;
   - utiliza argumentos válidos;
   - não inventa imóveis;
   - encerra a execução;
   - respeita o limite de etapas.
3. Crie mais três mensagens.
4. Registre pelo menos um erro ou comportamento inesperado.
5. Explique em qual camada o problema ocorreu:
   - interpretação;
   - escolha da ação;
   - argumentos;
   - ferramenta;
   - resposta final.


# Atividade prática — Parte 2

Crie uma terceira ferramenta.

Sugestões:

### Opção A — Consultar detalhes de um imóvel

Entrada:

```json
{
  "id_imovel": 2
}
```

Saída:

- todos os dados do imóvel;
- mensagem de não encontrado.

### Opção B — Comparar dois imóveis

Entrada:

```json
{
  "id_imovel_1": 1,
  "id_imovel_2": 2
}
```

Saída:

- diferença de preço;
- diferença de área;
- diferença de quartos;
- diferença de vagas.

### Opção C — Calcular custo inicial de locação

Use uma regra didática e claramente identificada como simulação, sem representar uma condição contratual real.

Para concluir:

1. crie o modelo Pydantic dos argumentos;
2. crie a função;
3. registre a ferramenta;
4. atualize a instrução do agente;
5. teste a ferramenta diretamente;
6. teste a ferramenta dentro do ciclo.


In [ ]:
# ESPAÇO PARA A NOVA FERRAMENTA DO GRUPO

# 1. Modelo Pydantic dos argumentos


# 2. Função Python


# 3. Registro em FERRAMENTAS


# 4. Atualização da descrição e da instrução


# 5. Testes


# Desafio opcional — Corrigir uma ação inválida

Atualmente, o agente encerra quando produz um JSON inválido.

Como desafio, implemente uma nova tentativa:

```text
Ação inválida
      ↓
Enviar ao modelo o erro de validação
      ↓
Pedir a correção do JSON
      ↓
Validar novamente
```

Defina um limite de apenas uma correção para evitar ciclos infinitos.


# O que construímos

Agora o protótipo consegue:

```text
Mensagem do cliente
        ↓
Interpretação com modelo aberto
        ↓
Escolha estruturada de uma ação
        ↓
Validação com Pydantic
        ↓
Execução de ferramenta Python
        ↓
Observação do resultado
        ↓
Nova decisão
        ↓
Resposta final
```

Isso já representa um **ciclo simples de agente com ferramentas**.

## Próxima evolução

Na próxima aula, poderemos reconstruir parte desse fluxo com uma biblioteca de agentes, comparando:

- código manual;
- abstrações prontas;
- facilidade de manutenção;
- controle do estado;
- observabilidade;
- limites e dependência do framework.


# Checklist de entrega

- [ ] base fictícia criada;
- [ ] ferramenta de consulta funcionando;
- [ ] ferramenta de cálculo funcionando;
- [ ] argumentos validados com Pydantic;
- [ ] registro autorizado de ferramentas;
- [ ] modelo escolhendo uma ação;
- [ ] resultado da ferramenta devolvido ao modelo;
- [ ] ciclo com limite máximo de etapas;
- [ ] histórico de execução disponível;
- [ ] pelo menos cinco cenários testados;
- [ ] um erro analisado;
- [ ] terceira ferramenta iniciada ou concluída.

## Pergunta de fechamento

Qual é a diferença entre:

1. o modelo sugerir uma ação;
2. o programa autorizar e executar essa ação?
